# Simulación del estómago — Fase 5: visualización (campos, vectores y animaciones)

**Qué hace este cuaderno.** Reúne los resultados de las fases anteriores y los convierte en figuras y animaciones listas para un artículo, una presentación o una exploración interactiva:
1. Un **módulo de funciones** reutilizable: mapas de color, flechas (`quiver`), líneas de corriente (`streamplot`) y campos derivados (vorticidad, criterio Q, presión, tensión).
2. Una **galería**: el mismo instante contado de cuatro maneras, una onda completa en fases, diagramas espacio–tiempo y trayectorias de partículas.
3. **Animaciones** con `matplotlib.animation`: en línea (HTML), en MP4 y en GIF.
4. **Plotly** (opcional): mapas con deslizador temporal, trayectorias y el registro de validación, todo interactivo y exportable a HTML.
5. **Exportación** de todo a una carpeta `figuras/`: PNG a 200 ppp, SVG vectorial, MP4, GIF y HTML.

**Datos de entrada**
- `campo_antro.npz` (fase 3 o 4). Si no está, o le faltan campos, el cuaderno vuelve a ejecutar el LBM de la fase 3 (≈ 75 s en CPU).
- `registro_validacion.csv` (fases 1b y 4), si existe. Si no, usa los valores de aquellas ejecuciones, copiados en la celda.

**Criterios de diseño**
- Mapa de color **secuencial** para magnitudes (rapidez, tensión) y **divergente, centrado en cero** para magnitudes con signo (vorticidad, criterio Q, presión).
- El sólido se dibuja en blanco, con la pared en gris oscuro.
- Las flechas se pueden **normalizar**: todas miden lo mismo y el color indica la rapidez. Así no desaparecen las lentas cuando hay mucho rango dinámico.

**Tiempo en Colab**: unos 3–4 minutos (más 75 s si hay que regenerar el flujo).

## 1. Configuración y carga de datos

In [ ]:
import math, time, os, glob
from dataclasses import dataclass, replace

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.collections import LineCollection
from matplotlib.colors import to_rgba
from IPython.display import HTML, display

plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (11, 4.2),
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "lines.linewidth": 2.0, "font.size": 10.5, "legend.frameon": False,
    "animation.embed_limit": 60,          # MB permitidos para animaciones en línea
})
GRIS = "#3b3b3b"; GRIS_MEDIO = "#9a9a96"; TINTA2 = "#52514e"
C1, C2, C3, C4, C5 = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#6250d6"
VERIFICACION = []

In [ ]:
CODIGO_LBM = r'''
# ===========================================================================
# Lattice Boltzmann D2Q9 escrito con funciones PURAS (sin modificar arrays en el sitio).
# Así el mismo código corre con NumPy, CuPy (GPU, misma API) o jax.numpy (GPU + jit).
# Unidades de red: dx = dt = 1, c_s^2 = 1/3, nu = (tau - 1/2)/3.
# Convenio de arrays: f tiene forma (9, ny, nx); la fila j = 0 es el fondo (y crece hacia arriba).
# ===========================================================================
CX  = [0, 1, 0, -1, 0, 1, -1, -1, 1]
CY  = [0, 0, 1, 0, -1, 1, 1, -1, -1]
W   = [4 / 9] + [1 / 9] * 4 + [1 / 36] * 4
OPP = [0, 3, 4, 1, 2, 7, 8, 5, 6]          # dirección opuesta


class Red:
    # Constantes D2Q9 ya convertidas al backend (xp) y al tipo de dato elegidos
    def __init__(self, xp, dtype="float64"):
        self.xp, self.dtype = xp, dtype
        self.cx = xp.asarray(CX, dtype=dtype).reshape(9, 1, 1)
        self.cy = xp.asarray(CY, dtype=dtype).reshape(9, 1, 1)
        self.w = xp.asarray(W, dtype=dtype).reshape(9, 1, 1)


def equilibrio(rho, ux, uy, R):
    cu = 3.0 * (R.cx * ux + R.cy * uy)
    usq = 1.5 * (ux * ux + uy * uy)
    return R.w * rho * (1.0 + cu + 0.5 * cu * cu - usq)


def macroscopicas(f, R, Fx=0.0, Fy=0.0):
    rho = f.sum(axis=0)
    jx = f[1] - f[3] + f[5] - f[6] - f[7] + f[8]        # sum_i f_i c_ix, escrito explícitamente (más rápido)
    jy = f[2] - f[4] + f[5] + f[6] - f[7] - f[8]
    ux = (jx + 0.5 * Fx) / rho                          # desplazamiento de Guo: u = (sum f c + F/2)/rho
    uy = (jy + 0.5 * Fy) / rho
    return rho, ux, uy


def colision_bgk(f, rho, ux, uy, tau, R, Fx=0.0, Fy=0.0):
    feq = equilibrio(rho, ux, uy, R)
    cu = R.cx * ux + R.cy * uy
    cF = R.cx * Fx + R.cy * Fy
    # término de fuerza de Guo (2002): segundo orden con fuerza volumétrica
    S = (1.0 - 0.5 / tau) * R.w * (3.0 * ((R.cx - ux) * Fx + (R.cy - uy) * Fy) + 9.0 * cu * cF)
    return f - (f - feq) / tau + S


def propagar(f, R):
    xp = R.xp
    return xp.stack([xp.roll(f[i], shift=(CY[i], CX[i]), axis=(0, 1)) for i in range(9)])


def rebote(fs, fpost, solido, uwx, uwy, rho, R):
    # Rebote a mitad de enlace con pared móvil (Ladd):
    # si la población i llega a un nodo fluido x desde un nodo sólido x - c_i, se sustituye por
    # f_i(x) = f*_opp(i)(x) + 6 w_i rho (c_i · u_w),  con u_w la velocidad de la pared en ese nodo sólido.
    xp = R.xp
    salida = []
    fluido = ~solido
    for i in range(9):
        if i == 0:
            salida.append(fs[0]); continue
        desde_solido = xp.roll(solido, shift=(CY[i], CX[i]), axis=(0, 1)) & fluido
        cu_w = 0.0
        if CX[i] != 0 and uwx is not None:
            cu_w = cu_w + CX[i] * xp.roll(uwx, shift=(CY[i], CX[i]), axis=(0, 1))
        if CY[i] != 0 and uwy is not None:
            cu_w = cu_w + CY[i] * xp.roll(uwy, shift=(CY[i], CX[i]), axis=(0, 1))
        rebotado = fpost[OPP[i]] + 6.0 * W[i] * rho * cu_w
        salida.append(xp.where(desde_solido, rebotado, fs[i]))
    return xp.stack(salida)


def rellenar(f, solido_viejo, solido_nuevo, rho, uwx, uwy, R):
    # Nodos que pasan de sólido a fluido ("nodos frescos") se inicializan con el equilibrio:
    # densidad = media de los vecinos fluidos, velocidad = la de la pared (Lallemand y Luo, 2003).
    xp = R.xp
    fresco = solido_viejo & ~solido_nuevo
    fluido_previo = (~solido_viejo).astype(rho.dtype)
    suma, n = 0.0 * rho, 0.0 * rho
    for i in range(1, 9):
        suma = suma + xp.roll(rho * fluido_previo, shift=(CY[i], CX[i]), axis=(0, 1))
        n = n + xp.roll(fluido_previo, shift=(CY[i], CX[i]), axis=(0, 1))
    rho_vec = xp.where(n > 0, suma / xp.maximum(n, 1.0), 1.0)
    cero = 0.0 * rho
    f_fresco = equilibrio(rho_vec, cero if uwx is None else uwx, cero if uwy is None else uwy, R)
    f = xp.where(fresco[None], f_fresco, f)
    # dentro del sólido guardamos el equilibrio en reposo, f_i = w_i (no interviene en la dinámica)
    return xp.where(solido_nuevo[None], R.w + 0.0 * f, f)


def crear_paso(R, tau, geometria, Fx=0.0, Fy=0.0):
    # Devuelve paso(f, t) -> f(t+1). geometria(t) -> (solido, uwx, uwy) en unidades de red.
    def paso(f, t):
        solido, uwx, uwy = geometria(t)
        rho, ux, uy = macroscopicas(f, R, Fx, Fy)
        fpost = colision_bgk(f, rho, ux, uy, tau, R, Fx, Fy)
        fs = propagar(fpost, R)
        fs = rebote(fs, fpost, solido, uwx, uwy, rho, R)
        solido2, uwx2, uwy2 = geometria(t + 1)
        return rellenar(fs, solido, solido2, rho, uwx2, uwy2, R)
    return paso


def tension_viscosa(f, tau, R, Fx=0.0, Fy=0.0):
    # Tensor de tensiones viscosas a partir de la parte de no equilibrio (sin derivar el campo):
    # sigma_ab = -(1 - 1/(2 tau)) sum_i (f_i - feq_i) c_ia c_ib    [unidades de red]
    rho, ux, uy = macroscopicas(f, R, Fx, Fy)
    fneq = f - equilibrio(rho, ux, uy, R)
    k = -(1.0 - 0.5 / tau)
    sxx = k * (fneq * R.cx * R.cx).sum(axis=0)
    syy = k * (fneq * R.cy * R.cy).sum(axis=0)
    sxy = k * (fneq * R.cx * R.cy).sum(axis=0)
    return sxx, syy, sxy


def geometria_fija(solido, R):
    xp = R.xp
    cero = xp.zeros(solido.shape, dtype=R.dtype)
    return lambda t: (solido, cero, cero)


# ---------------------------------------------------------------------------
# Paredes que se mueven: canal peristáltico periódico (test de Shapiro)
# ---------------------------------------------------------------------------
def geometria_peristaltica(nx, ny, a, phi, lam, c, R):
    # Semiancho H(x,t) = a (1 + phi sin(2 pi (x - c t)/lam)) alrededor del eje y = ny/2.
    # Las paredes se mueven solo en vertical (como en Shapiro et al., 1969).
    xp = R.xp
    x = xp.arange(nx, dtype=R.dtype)[None, :] + 0.5
    y = xp.arange(ny, dtype=R.dtype)[:, None] + 0.5
    yc = ny / 2.0
    k = 2 * math.pi / lam
    def geom(t):
        H = a * (1.0 + phi * xp.sin(k * (x - c * t)))
        dHdt = -a * phi * k * c * xp.cos(k * (x - c * t))
        abajo, arriba = y < yc - H, y > yc + H
        solido = abajo | arriba
        uwy = xp.where(abajo, -dHdt, 0.0 * y) + xp.where(arriba, dHdt, 0.0 * y)
        return solido, None, uwy                        # uwx = None: las paredes solo se mueven en vertical
    return geom


def integrar(paso, f, t0, n_pasos, R, usar_jax=False):
    # Bucle de tiempo. Con JAX se compila con lax.fori_loop (todo el bucle en la GPU).
    if usar_jax:
        from jax import lax
        f, _ = lax.fori_loop(0, n_pasos, lambda i, ft: (paso(ft[0], ft[1]), ft[1] + 1.0), (f, R.xp.asarray(t0, dtype=R.dtype)))
        return f
    for n in range(n_pasos):
        f = paso(f, t0 + n)
    return f

@dataclass
class ParamAntro:
    # Geometría y movimiento (unidades físicas: cm, s)
    Lx: float = 10.0          # cm   longitud (fondo -> píloro)
    H: float = 4.0            # cm   anchura en reposo
    margen: float = 1.0       # cm   holgura para que el fondo pueda dilatarse
    x_ini: float = 4.5        # cm   donde empiezan las ondas antrales
    x_plena: float = 6.5      # cm   donde alcanzan su amplitud máxima
    L_fondo: float = 3.5      # cm   zona proximal que se dilata para conservar el volumen
    A_max: float = 1.4        # cm   profundidad de la contracción en cada pared (70 % de oclusión)
    w: float = 0.5            # cm   semiancho de la onda
    c: float = 0.25           # cm/s velocidad de la onda peristáltica
    T: float = 20.0           # s    periodo (3 ondas por minuto)
    nu: float = 0.1           # cm²/s viscosidad cinemática (comida líquida espesa; agua = 0.01)
    rho: float = 1.0          # g/cm³
    # Discretización
    nx: int = 128             # nodos a lo largo (sin contar las paredes de los extremos)
    tau: float = 0.65


class Unidades:
    # Conversión física <-> red: dx fija la malla; dt sale de igualar la viscosidad.
    def __init__(self, pa):
        self.dx = pa.Lx / pa.nx                        # cm
        self.nu_red = (pa.tau - 0.5) / 3.0
        self.dt = self.nu_red * self.dx ** 2 / pa.nu    # s
        self.vel = self.dx / self.dt                    # (cm/s) por unidad de red
        self.tension = pa.rho * self.vel ** 2           # (dyn/cm² = 0.1 Pa) por unidad de red

    def resumen(self, pa):
        return {"dx [cm]": self.dx, "dt [s]": self.dt, "pasos por onda": pa.T / self.dt,
                "c onda [red]": pa.c / self.vel, "Re = c (H/2)/nu": pa.c * (pa.H / 2) / pa.nu,
                "nu [red]": self.nu_red}


def geometria_antro(pa, R):
    # Paredes superior e inferior con ondas que viajan hacia el píloro (x = Lx).
    # El fondo (x < L_fondo) se dilata lo justo para que el volumen total no cambie.
    # Extremos x = 0 (fondo) y x = Lx (píloro, cerrado en esta fase) son paredes fijas.
    xp = R.xp
    U = Unidades(pa)
    dx = U.dx
    ny = int(round((pa.H + 2 * pa.margen) / dx))
    nx = pa.nx + 2                                       # + una columna de pared en cada extremo
    x = (xp.arange(nx, dtype=R.dtype)[None, :] - 0.5) * dx          # cm (la columna 0 es pared)
    y = (xp.arange(ny, dtype=R.dtype)[:, None] + 0.5) * dx
    rampa = xp.clip((x - pa.x_ini) / (pa.x_plena - pa.x_ini), 0.0, 1.0)
    envolvente = pa.A_max * rampa * rampa * (3 - 2 * rampa)         # suave (smoothstep)
    ventana_fondo = xp.clip((pa.L_fondo + 1.0 - x) / 1.0, 0.0, 1.0) * (x > 0) * (x < pa.Lx)
    int_ventana = float(xp.sum(ventana_fondo)) * dx
    x0 = pa.x_ini - 3 * pa.w
    K = int(math.ceil((pa.Lx + 3 * pa.w - x0) / (pa.c * pa.T))) + 1
    extremos = (x <= 0) | (x >= pa.Lx)
    def ondas(t_fis):
        s = t_fis % pa.T
        d = 0.0 * x
        dd = 0.0 * x
        for k in range(K):
            xk = x0 + pa.c * (s + k * pa.T)
            g = xp.exp(-((x - xk) / pa.w) ** 2)
            d = d + g
            dd = dd + g * 2 * (x - xk) * pa.c / pa.w ** 2
        return envolvente * d, envolvente * dd                      # delta [cm], d delta/dt [cm/s]
    def geom(t):
        t_fis = t * U.dt
        delta, ddelta = ondas(t_fis)
        interior = (x > 0) & (x < pa.Lx)
        area = xp.sum(delta * interior) * dx                         # área contraída por pared
        darea = xp.sum(ddelta * interior) * dx
        d_f = area / int_ventana                                     # dilatación del fondo
        dd_f = darea / int_ventana
        yb = pa.margen + delta - ventana_fondo * d_f
        yt = pa.margen + pa.H - delta + ventana_fondo * d_f
        vb = ddelta - ventana_fondo * dd_f                           # cm/s
        vt = -ddelta + ventana_fondo * dd_f
        abajo, arriba = y < yb, y > yt
        solido = abajo | arriba | extremos
        uwy = (xp.where(abajo, vb, 0.0 * y) + xp.where(arriba, vt, 0.0 * y)) * ~extremos / U.vel
        return solido, None, uwy
    return geom, nx, ny, U

def a_numpy(a):
    # Convierte un array de cualquier backend (NumPy, CuPy, JAX) a NumPy
    if hasattr(a, "get"):
        return a.get()              # CuPy
    return np.asarray(a)            # NumPy y JAX


def interpolar_bilineal(campo, xs, ys):
    # campo (ny, nx) en nodos de red; xs, ys en coordenadas de red (centro del nodo i en x = i)
    ny, nx = campo.shape
    x = np.clip(xs, 0, nx - 1.001); y = np.clip(ys, 0, ny - 1.001)
    i0 = np.floor(x).astype(int); j0 = np.floor(y).astype(int)
    tx = x - i0; ty = y - j0
    c00 = campo[j0, i0]; c10 = campo[j0, i0 + 1]; c01 = campo[j0 + 1, i0]; c11 = campo[j0 + 1, i0 + 1]
    return (1 - tx) * (1 - ty) * c00 + tx * (1 - ty) * c10 + (1 - tx) * ty * c01 + tx * ty * c11


def simular_antro(pa, R, periodos=4, guardar_cada_s=0.25, n_particulas=3000, semilla=1,
                  usar_jax=False, informar=True):
    # Ejecuta el antro y devuelve instantáneas (en NumPy y unidades físicas) + trayectorias de trazadores.
    xp = R.xp
    geom, nx, ny, U = geometria_antro(pa, R)
    paso = crear_paso(R, pa.tau, geom)
    f = R.w * xp.ones((9, ny, nx), dtype=R.dtype)                     # reposo, rho = 1
    pasos_bloque = max(1, int(round(guardar_cada_s / U.dt)))
    n_bloques = int(round(periodos * pa.T / (pasos_bloque * U.dt)))
    if usar_jax:
        import jax
        avanzar = jax.jit(lambda f, t0: integrar(paso, f, t0, pasos_bloque, R, usar_jax=True))
    else:
        avanzar = lambda f, t0: integrar(paso, f, t0, pasos_bloque, R)
    # trazadores: repartidos al azar por el fluido inicial (coordenadas de red)
    rng = np.random.default_rng(semilla)
    s0 = a_numpy(geom(0)[0])
    cand = np.argwhere(~s0)
    elegido = cand[rng.choice(len(cand), n_particulas, replace=False)]
    py = elegido[:, 0] + rng.uniform(-0.5, 0.5, n_particulas)
    px = elegido[:, 1] + rng.uniform(-0.5, 0.5, n_particulas)
    hist = {k: [] for k in ("t", "ux", "uy", "rho", "solido", "tau_max", "px", "py", "masa")}
    def campos(f, t):
        rho, ux, uy = macroscopicas(f, R)
        sxx, syy, sxy = tension_viscosa(f, pa.tau, R)
        tmax = xp.sqrt(0.25 * (sxx - syy) ** 2 + sxy ** 2)            # cizalla máxima
        sol = geom(t)[0]
        return [a_numpy(v) for v in (rho, ux, uy, tmax, sol)]
    t_red = 0
    rho, ux, uy, tmax, sol = campos(f, t_red)
    t_ini = time.time()
    for b in range(n_bloques + 1):
        hist["t"].append(t_red * U.dt)
        hist["ux"].append(np.where(sol, np.nan, ux) * U.vel)
        hist["uy"].append(np.where(sol, np.nan, uy) * U.vel)
        hist["rho"].append(np.where(sol, np.nan, rho))
        hist["tau_max"].append(np.where(sol, np.nan, tmax) * U.tension / 10.0)   # Pa
        hist["solido"].append(sol)
        hist["px"].append(px.copy()); hist["py"].append(py.copy())
        hist["masa"].append(float(np.sum(np.where(sol, 0.0, rho))))
        if b == n_bloques:
            break
        f = avanzar(f, float(t_red))
        t_red += pasos_bloque
        rho2, ux2, uy2, tmax2, sol2 = campos(f, t_red)
        # trazadores: Heun con el campo al principio y al final del bloque (desplazamiento en nodos)
        n_p = pasos_bloque
        vx1 = interpolar_bilineal(np.where(sol, 0, ux), px, py); vy1 = interpolar_bilineal(np.where(sol, 0, uy), px, py)
        qx, qy = px + n_p * vx1, py + n_p * vy1
        vx2 = interpolar_bilineal(np.where(sol2, 0, ux2), qx, qy); vy2 = interpolar_bilineal(np.where(sol2, 0, uy2), qx, qy)
        nx_, ny_ = px + 0.5 * n_p * (vx1 + vx2), py + 0.5 * n_p * (vy1 + vy2)
        ii = np.clip(np.rint(nx_).astype(int), 0, nx - 1); jj = np.clip(np.rint(ny_).astype(int), 0, ny - 1)
        dentro = ~sol2[jj, ii]
        px = np.where(dentro, nx_, px); py = np.where(dentro, ny_, py)   # si caería en la pared, se queda
        rho, ux, uy, tmax, sol = rho2, ux2, uy2, tmax2, sol2
        if informar and (b + 1) % max(1, n_bloques // 8) == 0:
            print(f"  t = {t_red * U.dt:6.1f} s   ({time.time() - t_ini:5.1f} s de cálculo)")
    out = {k: np.array(v) for k, v in hist.items()}
    out["U"], out["nx"], out["ny"], out["pasos"] = U, nx, ny, t_red
    out["segundos"] = time.time() - t_ini
    return out



'''

In [ ]:
CLAVES = ("t", "ux", "uy", "solido", "x", "y", "dx", "periodo")

def regenerar_campo(periodos=2):
    exec(CODIGO_LBM, globals())
    pa = ParamAntro(); R = Red(np)
    S = simular_antro(pa, R, periodos=periodos, n_particulas=10, informar=False)
    t = S["t"]; k = int(round(pa.T / (t[1] - t[0]))); ult = slice(len(t) - k - 1, len(t))
    dx = S["U"].dx
    return dict(t=t[ult] - t[ult][0], ux=np.nan_to_num(S["ux"][ult]), uy=np.nan_to_num(S["uy"][ult]),
                solido=S["solido"][ult], x=(np.arange(S["nx"]) - 0.5) * dx, y=(np.arange(S["ny"]) + 0.5) * dx,
                dx=dx, periodo=pa.T, nu=pa.nu, tau_max=np.nan_to_num(S["tau_max"][ult]),
                rho=np.where(S["solido"][ult], 1.0, np.nan_to_num(S["rho"][ult], nan=1.0)))

datos = dict(np.load("campo_antro.npz")) if os.path.exists("campo_antro.npz") else {}
faltan = [c for c in CLAVES + ("tau_max", "rho") if c not in datos]
if faltan:
    print(f"Regenerando el flujo con LBM (faltan: {', '.join(faltan) or 'archivo'}) ... ≈ 75 s")
    t0 = time.time(); datos = regenerar_campo(); print(f"  hecho en {time.time() - t0:.0f} s")
    np.savez_compressed("campo_antro.npz", **datos)
else:
    print("Usando campo_antro.npz existente")
print(f"{len(datos['t'])} marcos, malla {datos['ux'].shape[2]}×{datos['ux'].shape[1]}, Δx = {float(datos['dx']):.4f} cm, periodo {float(datos['periodo']):.0f} s")

## 2. Módulo de visualización

In [ ]:
# ===========================================================================
# Módulo de visualización del estómago 2D. Todas las funciones reciben un `Campo`
# (los datos de la fase 3) y un índice de marco k, y dibujan sobre un eje de Matplotlib.
# ===========================================================================
CMAPS = {
    "rapidez": "magma",      # magnitud (secuencial)
    "vorticidad": "RdBu_r",  # signo: giro antihorario (rojo) / horario (azul) -> divergente y simétrica
    "Q": "PiYG",             # Q > 0 vórtice, Q < 0 deformación -> divergente
    "presion": "PuOr_r",     # sobre/depresión -> divergente
    "tension": "inferno",    # magnitud (secuencial)
}
COLOR_PARED = "#3b3b3b"


class Campo:
    # Envuelve los arrays exportados por la fase 3 y calcula los campos derivados bajo demanda.
    def __init__(self, datos):
        self.t = np.asarray(datos["t"]); self.ux = datos["ux"]; self.uy = datos["uy"]
        self.sol = datos["solido"].astype(bool)
        self.x = np.asarray(datos["x"]); self.y = np.asarray(datos["y"])
        self.dx = float(datos["dx"]); self.T = float(datos["periodo"])
        self.tau = datos.get("tau_max"); self.rho = datos.get("rho")
        self.X, self.Y = np.meshgrid(self.x, self.y)
        self.extent = [self.x[0] - self.dx / 2, self.x[-1] + self.dx / 2, self.y[0] - self.dx / 2, self.y[-1] + self.dx / 2]

    def __len__(self):
        return len(self.t)

    def mascara(self, Z, k):
        return np.where(self.sol[k], np.nan, Z)

    def rapidez(self, k):
        return self.mascara(np.hypot(self.ux[k], self.uy[k]), k)

    def gradientes(self, k):
        u, v = self.ux[k], self.uy[k]
        return (np.gradient(u, self.dx, axis=1), np.gradient(u, self.dx, axis=0),
                np.gradient(v, self.dx, axis=1), np.gradient(v, self.dx, axis=0))

    def vorticidad(self, k):
        ux_, uy_, vx_, vy_ = self.gradientes(k)
        return self.mascara(vx_ - uy_, k)

    def criterio_Q(self, k):
        # Q = ½(|Ω|² − |S|²). En 2D: Q = −½(u_x² + v_y²) − u_y v_x  (Q > 0: domina la rotación -> núcleo de vórtice)
        ux_, uy_, vx_, vy_ = self.gradientes(k)
        return self.mascara(-0.5 * (ux_ ** 2 + vy_ ** 2) - uy_ * vx_, k)

    def presion(self, k):
        # p - p_media = c_s² (rho - rho_media) en unidades de red -> se devuelve relativa (adimensional)
        if self.rho is None:
            return None
        r = self.mascara(self.rho[k], k)
        return (r - np.nanmean(r)) / 3.0

    def tension(self, k):
        return None if self.tau is None else self.mascara(self.tau[k], k)


def limites(Z, simetrico=False, pct=99.0):
    a = np.nanpercentile(np.abs(Z) if simetrico else Z, pct)
    return (-a, a) if simetrico else (0.0, a)


def mapa(ax, campo, Z, k, tipo, etiqueta=None, vlim=None, barra=True):
    # Mapa de color de un campo escalar con el sólido en blanco y el contorno de la pared
    simetrico = tipo in ("vorticidad", "Q", "presion")
    vmin, vmax = vlim if vlim is not None else limites(Z, simetrico)
    im = ax.imshow(Z, origin="lower", extent=campo.extent, cmap=CMAPS[tipo], vmin=vmin, vmax=vmax,
                   aspect="equal", interpolation="nearest")
    pared(ax, campo, k)
    if barra:
        plt.colorbar(im, ax=ax, shrink=0.8, pad=0.01, label=etiqueta or tipo)
    return im


def pared(ax, campo, k, **kw):
    kw = {"colors": COLOR_PARED, "linewidths": 1.0, **kw}
    return ax.contour(campo.X, campo.Y, campo.sol[k].astype(float), levels=[0.5], **kw)


def flechas(ax, campo, k, cada=5, normalizar=True, color_por_rapidez=True, escala=None, vmax=None):
    # quiver: con 'normalizar' todas las flechas miden lo mismo y el color da la rapidez.
    # Evita el problema típico de quiver: con mucho rango dinámico, las flechas lentas desaparecen.
    s = (slice(None, None, cada), slice(None, None, cada))
    U, V = campo.ux[k][s], campo.uy[k][s]
    m = np.hypot(U, V)
    fluido = ~campo.sol[k][s] & (m > 0)
    Xs, Ys = campo.X[s][fluido], campo.Y[s][fluido]
    U, V, m = U[fluido], V[fluido], m[fluido]
    if normalizar:
        U, V = U / m, V / m
        escala = escala or 1.2 / (cada * campo.dx)
    args = (Xs, Ys, U, V) + ((m,) if color_por_rapidez else ())
    q = ax.quiver(*args, cmap=CMAPS["rapidez"] if color_por_rapidez else None,
                  clim=(0, vmax or np.nanpercentile(m, 99)) if color_por_rapidez else None,
                  color=None if color_por_rapidez else "#2a78d6", scale=escala, scale_units="xy" if normalizar else None,
                  width=0.0035, headwidth=3.5, headlength=4, pivot="mid")
    return q


def corrientes(ax, campo, k, densidad=1.2, ancho_por_rapidez=True, color="#2a78d6"):
    # streamplot con el grosor de línea proporcional a la rapidez
    u = np.where(campo.sol[k], 0.0, campo.ux[k]); v = np.where(campo.sol[k], 0.0, campo.uy[k])
    sp = np.hypot(u, v)
    lw = 0.4 + 2.2 * sp / (np.nanpercentile(sp, 99) + 1e-30) if ancho_por_rapidez else 0.8
    return ax.streamplot(campo.x, campo.y, u, v, density=densidad, color=color, linewidth=lw, arrowsize=0.8,
                         broken_streamlines=True)


def preparar_eje(ax, campo, titulo=None, ejes=True):
    ax.set_xlim(campo.extent[0], campo.extent[1]); ax.set_ylim(campo.extent[2], campo.extent[3])
    ax.set_aspect("equal"); ax.grid(False)
    if titulo:
        ax.set_title(titulo, fontsize=10)
    if ejes:
        ax.set_xlabel("x [cm]"); ax.set_ylabel("y [cm]")
    else:
        ax.set_xticks([]); ax.set_yticks([])


def guardar_animacion(anim, nombre, fps=12, dpi=110):
    # Intenta MP4 (ffmpeg) y GIF (pillow). Devuelve las rutas creadas.
    rutas = []
    os.makedirs("figuras", exist_ok=True)
    try:
        ruta = f"figuras/{nombre}.mp4"
        anim.save(ruta, writer=animation.FFMpegWriter(fps=fps, bitrate=1800), dpi=dpi)
        rutas.append(ruta)
    except Exception as e:
        print(f"  MP4 no disponible ({type(e).__name__}): se omite")
    try:
        ruta = f"figuras/{nombre}.gif"
        anim.save(ruta, writer=animation.PillowWriter(fps=fps), dpi=max(60, dpi - 40))
        rutas.append(ruta)
    except Exception as e:
        print(f"  GIF no disponible ({type(e).__name__}): se omite")
    return rutas


def guardar_figura(fig, nombre):
    os.makedirs("figuras", exist_ok=True)
    fig.savefig(f"figuras/{nombre}.png", dpi=200, bbox_inches="tight")
    fig.savefig(f"figuras/{nombre}.svg", bbox_inches="tight")

### 2.1 Verificación de los campos derivados
Antes de interpretar mapas de vorticidad o del criterio Q, comprobamos las fórmulas con campos de solución conocida:
- **Rotación sólida** $(u, v) = (-\Omega y, \Omega x)$: vorticidad $2\Omega$, $Q = \Omega^2 > 0$.
- **Deformación pura** $(u, v) = (\epsilon x, -\epsilon y)$: vorticidad 0, $Q = -\epsilon^2 < 0$.

El criterio Q separa lo que **gira** (núcleo de un vórtice, $Q>0$) de lo que solo **se estira** ($Q<0$). La vorticidad sola no distingue un vórtice de una capa de cizalla.

In [ ]:
xs = np.linspace(-1, 1, 41); Xs, Ys = np.meshgrid(xs, xs)
base = dict(t=np.array([0.0]), solido=np.zeros((1, 41, 41), bool), x=xs, y=xs, dx=xs[1] - xs[0], periodo=1.0)
Om, eps = 0.7, 0.4
rot = Campo({**base, "ux": (-Om * Ys)[None], "uy": (Om * Xs)[None]})
est = Campo({**base, "ux": (eps * Xs)[None], "uy": (-eps * Ys)[None]})
errs = {"vorticidad rotación": np.nanmax(np.abs(rot.vorticidad(0) - 2 * Om)),
        "Q rotación": np.nanmax(np.abs(rot.criterio_Q(0) - Om ** 2)),
        "vorticidad deformación": np.nanmax(np.abs(est.vorticidad(0))),
        "Q deformación": np.nanmax(np.abs(est.criterio_Q(0) + eps ** 2))}
for k_, v_ in errs.items():
    VERIFICACION.append({"test": f"V1 {k_}", "valor": float(v_), "criterio": "< 1e-12", "pasa": v_ < 1e-12})
print({k_: f"{v_:.1e}" for k_, v_ in errs.items()})

## 3. Galería

### 3.1 El mismo instante, cuatro maneras de verlo
Elegimos el momento en que el chorro retropulsivo es más intenso: el marco con la velocidad axial negativa más fuerte en el antro.

In [ ]:
C = Campo(datos)
xa = C.x > 7.0
k_jet = int(np.argmin([np.nanmin(np.where(C.sol[k][:, xa], np.nan, C.ux[k][:, xa])) for k in range(len(C))]))
vmax_sp = float(np.nanpercentile([C.rapidez(k) for k in range(len(C))], 99.5))
print(f"Marco del chorro máximo: k = {k_jet} (t = {C.t[k_jet]:.1f} s de la onda)")

fig, axs = plt.subplots(2, 2, figsize=(14, 7.2))
a = axs[0, 0]
flechas(a, C, k_jet, cada=4, normalizar=False, color_por_rapidez=False, escala=4.0)
pared(a, C, k_jet); preparar_eje(a, C, "quiver directo: las zonas lentas 'desaparecen'")
a = axs[0, 1]
mapa(a, C, C.rapidez(k_jet), k_jet, "rapidez", "rapidez [cm/s]", vlim=(0, vmax_sp))
flechas(a, C, k_jet, cada=5, normalizar=True, color_por_rapidez=False)
preparar_eje(a, C, "rapidez (color) + dirección (flechas normalizadas)")
a = axs[1, 0]
mapa(a, C, C.rapidez(k_jet), k_jet, "rapidez", "rapidez [cm/s]", vlim=(0, vmax_sp))
corrientes(a, C, k_jet, color="white")
preparar_eje(a, C, "líneas de corriente (grosor ∝ rapidez)")
a = axs[1, 1]
w = C.vorticidad(k_jet); Q = C.criterio_Q(k_jet)
mapa(a, C, w, k_jet, "vorticidad", "vorticidad [1/s]")
a.contour(C.X, C.Y, np.nan_to_num(Q), levels=[np.nanpercentile(Q[Q > 0], 75)], colors="k", linewidths=1.0, linestyles="--")
preparar_eje(a, C, "vorticidad + núcleos de vórtice (Q > 0, discontinua)")
fig.suptitle("Antro: chorro retropulsivo visto de cuatro maneras", fontweight="bold")
fig.tight_layout(); guardar_figura(fig, "01_cuatro_maneras"); plt.show()

**Qué enseña cada panel.**
- El `quiver` directo es engañoso con mucho rango dinámico: solo se ven las flechas del chorro.
- Normalizar las flechas y llevar la rapidez al color separa dirección e intensidad.
- Las líneas de corriente muestran la **topología**: dónde recircula el fluido.
- La vorticidad muestra **dónde gira** el fluido. Con la línea discontinua del criterio Q, distinguimos los núcleos de vórtice de las capas de cizalla junto a la pared, que también tienen vorticidad pero no giran.

### 3.2 Una onda completa: cinco magnitudes en cuatro fases

In [ ]:
fases = np.linspace(0, len(C) - 1, 5)[:-1].astype(int)
magnitudes = [("rapidez", lambda k: C.rapidez(k), "cm/s"), ("vorticidad", lambda k: C.vorticidad(k), "1/s"),
              ("Q", lambda k: C.criterio_Q(k), "1/s²"), ("presion", lambda k: C.presion(k), "unid. de red"),
              ("tension", lambda k: C.tension(k), "Pa")]
magnitudes = [m for m in magnitudes if m[1](0) is not None]
fig, axs = plt.subplots(len(magnitudes), 4, figsize=(15, 2.3 * len(magnitudes) + 0.6))
for f, (tipo, fun, unidad) in enumerate(magnitudes):
    todos = np.array([fun(k) for k in fases])
    simetrico = tipo in ("vorticidad", "Q", "presion")
    vl = limites(todos, simetrico, 99.5)
    for c, k in enumerate(fases):
        a = axs[f, c]
        im = mapa(a, C, fun(k), k, tipo, vlim=vl, barra=False)
        if tipo == "rapidez":
            flechas(a, C, k, cada=6, normalizar=True, color_por_rapidez=False)
        preparar_eje(a, C, f"t = {C.t[k]:.0f} s" if f == 0 else None, ejes=False)
        if c == 0:
            a.set_ylabel(tipo)
    fig.colorbar(im, ax=axs[f, :], shrink=0.85, pad=0.01, label=f"{tipo} [{unidad}]")
fig.suptitle("Una onda antral completa (el píloro, cerrado, está a la derecha)", fontweight="bold")
guardar_figura(fig, "02_onda_completa"); plt.show()

### 3.3 Diagramas espacio–tiempo
Un diagrama espacio–tiempo apila en vertical el perfil de una magnitud a lo largo del eje del estómago, instante tras instante. Las ondas aparecen como bandas inclinadas, cuya pendiente es la velocidad de la onda, y el chorro retropulsivo como una banda de signo opuesto.

In [ ]:
j_eje = C.sol.shape[1] // 2
u_eje = np.array([np.where(C.sol[k][j_eje], np.nan, C.ux[k][j_eje]) for k in range(len(C))])
hueco = (~C.sol).sum(axis=1) * C.dx
tz = np.array([np.nanmax(C.tension(k), axis=0) for k in range(len(C))]) if C.tau is not None else None
paneles = [(u_eje, "RdBu_r", True, "u axial en el eje [cm/s]"), (hueco, "Greys_r", False, "luz del estómago [cm]")]
if tz is not None:
    paneles.append((tz, "inferno", False, "cizalla máx. en la sección [Pa]"))
fig, axs = plt.subplots(1, len(paneles), figsize=(5 * len(paneles), 4.2))
for a, (Z, cm, sim, lab) in zip(axs, paneles):
    vmin, vmax = limites(Z, sim, 99.5) if sim else (np.nanmin(Z), np.nanpercentile(Z, 99.5))
    im = a.imshow(Z, origin="lower", aspect="auto", cmap=cm, vmin=vmin, vmax=vmax, extent=[C.x[0], C.x[-1], C.t[0], C.t[-1]])
    plt.colorbar(im, ax=a, label=lab); a.grid(False)
    a.set(xlabel="x: fondo → píloro [cm]", ylabel="t [s]")
fig.suptitle("Diagramas espacio–tiempo de una onda", fontweight="bold")
fig.tight_layout(); guardar_figura(fig, "03_espacio_tiempo"); plt.show()

### 3.4 Trayectorias de partículas de comida
Partículas de 0.5, 2 y 4 mm (densidad 1.03 g/cm³), con el modelo de arrastre y sedimentación de la fase 4, seguidas durante 3 ondas. El color de cada trayectoria indica el tiempo, de claro a oscuro.

In [ ]:
def a_numpy(a):
    # Convierte un array de cualquier backend (NumPy, CuPy, JAX) a NumPy
    if hasattr(a, "get"):
        return a.get()              # CuPy
    return np.asarray(a)            # NumPy y JAX


def interpolar_bilineal(campo, xs, ys):
    # campo (ny, nx) en nodos de red; xs, ys en coordenadas de red (centro del nodo i en x = i)
    ny, nx = campo.shape
    x = np.clip(xs, 0, nx - 1.001); y = np.clip(ys, 0, ny - 1.001)
    i0 = np.floor(x).astype(int); j0 = np.floor(y).astype(int)
    tx = x - i0; ty = y - j0
    c00 = campo[j0, i0]; c10 = campo[j0, i0 + 1]; c01 = campo[j0 + 1, i0]; c11 = campo[j0 + 1, i0 + 1]
    return (1 - tx) * (1 - ty) * c00 + tx * (1 - ty) * c10 + (1 - tx) * ty * c01 + tx * ty * c11

In [ ]:
class CampoPeriodico:
    # Campo de velocidades de un periodo de onda (fase 3), repetido periódicamente en el tiempo.
    def __init__(self, datos):
        self.t = datos["t"]; self.ux = datos["ux"]; self.uy = datos["uy"]; self.sol = datos["solido"]
        self.x = datos["x"]; self.y = datos["y"]; self.dx = float(datos["dx"]); self.T = float(datos["periodo"])
        self.nu = float(datos["nu"])
        self.tau = datos["tau_max"] if "tau_max" in datos else None
        self.nf = len(self.t) - 1                    # el último marco repite el primero (t = T)
        self.dtf = self.t[1] - self.t[0]
        # hueco local (cm) en cada columna y marco: nº de nodos fluidos × dx
        self.hueco = (~self.sol).sum(axis=1) * self.dx

    def _marco(self, t):
        s = (t % self.T) / self.dtf
        k = int(np.floor(s)) % self.nf
        return k, s - np.floor(s)

    def velocidad(self, t, px, py):
        k, a = self._marco(t)
        u0 = self._interp(self.ux[k], px, py); u1 = self._interp(self.ux[k + 1], px, py)
        v0 = self._interp(self.uy[k], px, py); v1 = self._interp(self.uy[k + 1], px, py)
        return (1 - a) * u0 + a * u1, (1 - a) * v0 + a * v1

    def _indices(self, px, py):
        fx = (px - self.x[0]) / self.dx; fy = (py - self.y[0]) / self.dx
        return fx, fy

    def _interp(self, campo, px, py):
        fx, fy = self._indices(px, py)
        return interpolar_bilineal(campo, fx, fy)

    def solido_en(self, t, px, py):
        k, a = self._marco(t)
        sol = self.sol[k if a < 0.5 else k + 1]
        fx, fy = self._indices(px, py)
        i = np.clip(np.rint(fx).astype(int), 0, sol.shape[1] - 1); j = np.clip(np.rint(fy).astype(int), 0, sol.shape[0] - 1)
        return sol[j, i], sol, i, j

    def hueco_en(self, t, px):
        k, _ = self._marco(t)
        fx = (px - self.x[0]) / self.dx
        i = np.clip(np.rint(fx).astype(int), 0, self.hueco.shape[1] - 1)
        return self.hueco[k, i]


def tiempo_relajacion(d_cm, rho_p, rho_f, nu, v_rel):
    # Arrastre de Stokes con corrección de Schiller–Naumann: tau = rho_p d² / (18 mu f(Re_p))
    mu = rho_f * nu
    Re_p = np.abs(v_rel) * d_cm / nu
    f = 1 + 0.15 * Re_p ** 0.687
    return rho_p * d_cm ** 2 / (18 * mu * f)


def seguir_particulas(campo, d_mm, rho_p, rho_f=1.0, g=981.0, t_final=200.0, dt=0.02, semilla=0,
                      x_zona=8.0, frac_cuello=0.6, guardar_cada=0.5):
    # Partículas con inercia y sedimentación (gravedad hacia -y) en el campo periódico.
    # Integración exponencial con el fluido congelado durante dt (exacta para arrastre lineal).
    # Evento de compresión: la partícula está en el antro terminal (x > x_zona) dentro de un cuello
    # con hueco < frac_cuello · H. Se cuenta una vez por paso de onda.
    rng = np.random.default_rng(semilla)
    d_mm = np.asarray(d_mm, float); n = d_mm.size; d_cm = d_mm / 10
    s0 = campo.sol[0]
    cand = np.argwhere(~s0)
    sel = cand[rng.choice(len(cand), n, replace=True)]
    py = campo.y[sel[:, 0]] + rng.uniform(-0.5, 0.5, n) * campo.dx
    px = campo.x[sel[:, 1]] + rng.uniform(-0.5, 0.5, n) * campo.dx
    vx, vy = campo.velocidad(0.0, px, py)
    flot = (1 - rho_f / rho_p) * g
    cuerpo = (campo.x > 3.8) & (campo.x < 4.4)                              # cuerpo gástrico: sin ondas ni dilatación
    H_ref = float(np.median(campo.hueco[:, cuerpo]))
    ultimo_evento = np.full(n, -1e9)
    eventos = np.zeros(n)
    tiempo_zona = np.zeros(n)
    hist_x, hist_y, hist_t = [px.copy()], [py.copy()], [0.0]
    cada = max(1, int(round(guardar_cada / dt)))
    for paso in range(int(round(t_final / dt))):
        t = paso * dt
        ux, uy = campo.velocidad(t, px, py)
        rx, ry = vx - ux, vy - uy
        tau = tiempo_relajacion(d_cm, rho_p, rho_f, campo.nu, np.hypot(rx, ry))
        vtx, vty = 0.0, -tau * flot                                          # velocidad terminal relativa
        e = np.exp(-dt / tau)
        cx_, cy_ = vx - ux - vtx, vy - uy - vty
        nx_ = px + (ux + vtx) * dt + cx_ * tau * (1 - e)
        ny_ = py + (uy + vty) * dt + cy_ * tau * (1 - e)
        vx = ux + vtx + cx_ * e; vy = uy + vty + cy_ * e
        # paredes: si el centro entra en el sólido, se recoloca en el nodo fluido más cercano de su columna
        dentro, sol, i, j = campo.solido_en(t + dt, nx_, ny_)
        if dentro.any():
            for q in np.where(dentro)[0]:
                col = ~sol[:, i[q]]
                if col.any():
                    jj = np.where(col)[0]
                    jn = jj[np.argmin(np.abs(jj - j[q]))]
                    ny_[q] = campo.y[jn]
                else:
                    nx_[q], ny_[q] = px[q], py[q]
                vy[q] = 0.0
        px, py = nx_, ny_
        # eventos de compresión en el antro terminal
        en_zona = px > x_zona
        tiempo_zona += en_zona * dt
        cuello = en_zona & (campo.hueco_en(t + dt, px) < frac_cuello * H_ref)
        nuevo = cuello & (t - ultimo_evento > 0.5 * campo.T)
        eventos += nuevo; ultimo_evento = np.where(nuevo, t, ultimo_evento)
        if (paso + 1) % cada == 0:
            hist_x.append(px.copy()); hist_y.append(py.copy()); hist_t.append(t + dt)
    return {"x": np.array(hist_x), "y": np.array(hist_y), "t": np.array(hist_t), "d_mm": d_mm,
            "eventos_por_min": eventos / (t_final / 60.0), "frac_tiempo_zona": tiempo_zona / t_final}

In [ ]:
CP = CampoPeriodico(datos)
t0 = time.time()
TRAY = seguir_particulas(CP, np.repeat([0.5, 2.0, 4.0], 120), 1.03, t_final=3 * CP.T, dt=0.02, guardar_cada=0.25, semilla=4)
print(f"Seguimiento de 360 partículas durante 3 ondas: {time.time() - t0:.1f} s")
COL_D = {0.5: C1, 2.0: C3, 4.0: C2}

fig, axs = plt.subplots(1, 3, figsize=(15, 3.8))
for a, dv in zip(axs, (0.5, 2.0, 4.0)):
    m = np.where(TRAY["d_mm"] == dv)[0][:40]
    base_rgba = np.array(to_rgba(COL_D[dv]))
    for q in m:
        xs_, ys_ = TRAY["x"][:, q], TRAY["y"][:, q]
        n = len(xs_)
        cols = np.repeat(base_rgba[None], n - 1, axis=0); cols[:, 3] = np.linspace(0.15, 0.9, n - 1)
        a.add_collection(LineCollection(np.stack([np.column_stack([xs_[:-1], ys_[:-1]]),
                                                                          np.column_stack([xs_[1:], ys_[1:]])], axis=1),
                                                                colors=cols, linewidths=0.9))
    a.scatter(TRAY["x"][-1, m], TRAY["y"][-1, m], s=5 * dv + 4, color=COL_D[dv], zorder=3)
    pared(a, C, (len(TRAY["t"]) - 1) % (len(C) - 1))
    preparar_eje(a, C, f"d = {dv} mm")
fig.suptitle("Trayectorias durante 3 ondas (el tono se intensifica con el tiempo)", fontweight="bold")
fig.tight_layout(); guardar_figura(fig, "04_trayectorias"); plt.show()

## 4. Animaciones con Matplotlib

**Claves de rendimiento.** No se redibuja la figura entera en cada marco: se actualizan solo los objetos.
- `imshow` → `set_data`.
- `quiver` → `set_UVC`.
- `scatter` → `set_offsets`.

Las líneas de corriente no se pueden actualizar así, y se recalculan en cada marco, lo que hace esa animación más lenta. Cada animación se muestra en línea (HTML) y se guarda en MP4 (ffmpeg) y GIF (pillow) en `figuras/`.

### A1 — Rapidez y dirección del flujo

In [ ]:
def anim_flujo(C, cada=5, fps=12, paso=1):
    fig, a = plt.subplots(figsize=(9, 4.2))
    k0 = 0
    im = mapa(a, C, C.rapidez(k0), k0, "rapidez", "rapidez [cm/s]", vlim=(0, vmax_sp))
    s = (slice(None, None, cada), slice(None, None, cada))
    Xq, Yq = C.X[s], C.Y[s]
    def uv(k):
        U, V = C.ux[k][s], C.uy[k][s]; m = np.hypot(U, V) + 1e-12
        f = ~C.sol[k][s]
        return np.where(f, U / m, 0.0), np.where(f, V / m, 0.0)
    U0, V0 = uv(k0)
    q = a.quiver(Xq, Yq, U0, V0, color="white", scale=1.2 / (cada * C.dx), scale_units="xy", width=0.003, pivot="mid")
    fondo_solido = a.imshow(np.where(C.sol[k0], 1.0, np.nan), origin="lower", extent=C.extent, cmap="Greys", vmin=0, vmax=1.5, aspect="equal")
    preparar_eje(a, C); tit = a.set_title("")
    def frame(k):
        im.set_data(C.rapidez(k)); q.set_UVC(*uv(k)); fondo_solido.set_data(np.where(C.sol[k], 1.0, np.nan))
        tit.set_text(f"Flujo en el antro — t = {C.t[k]:.1f} s de la onda")
        return im, q, fondo_solido, tit
    an = animation.FuncAnimation(fig, frame, frames=range(0, len(C), paso), interval=1000 / fps, blit=True)
    plt.close(fig)
    return an

t0 = time.time()
A1 = anim_flujo(C)
display(HTML(A1.to_jshtml()))
print("Guardado:", guardar_animacion(A1, "A1_flujo"), f"({time.time() - t0:.0f} s)")

### A2 — Vorticidad con las partículas de comida encima

In [ ]:
def anim_vorticidad_particulas(C, TRAY, fps=12):
    fig, a = plt.subplots(figsize=(9, 4.2))
    vl = limites(np.array([C.vorticidad(k) for k in range(0, len(C), 4)]), True, 99)
    nf = len(C) - 1
    k_de = lambda n: int(round(TRAY["t"][n] / C.T * nf)) % nf
    im = mapa(a, C, C.vorticidad(k_de(0)), k_de(0), "vorticidad", "vorticidad [1/s]", vlim=vl)
    fondo = a.imshow(np.where(C.sol[0], 1.0, np.nan), origin="lower", extent=C.extent, cmap="Greys", vmin=0, vmax=1.5, aspect="equal")
    colores = [COL_D[d] for d in TRAY["d_mm"]]
    sc = a.scatter(TRAY["x"][0], TRAY["y"][0], s=3 * TRAY["d_mm"] + 3, c=colores, edgecolors="white", linewidths=0.3, zorder=3)
    for dv in COL_D:
        a.scatter([], [], s=3 * dv + 3, color=COL_D[dv], label=f"{dv} mm")
    a.legend(loc="upper left", fontsize=8, ncol=3)
    preparar_eje(a, C); tit = a.set_title("")
    def frame(n):
        k = k_de(n)
        im.set_data(C.vorticidad(k)); fondo.set_data(np.where(C.sol[k], 1.0, np.nan))
        sc.set_offsets(np.column_stack([TRAY["x"][n], TRAY["y"][n]]))
        tit.set_text(f"Vorticidad y partículas — t = {TRAY['t'][n]:.1f} s")
        return im, fondo, sc, tit
    an = animation.FuncAnimation(fig, frame, frames=range(0, len(TRAY["t"]), 2), interval=1000 / fps, blit=True)
    plt.close(fig)
    return an

t0 = time.time()
A2 = anim_vorticidad_particulas(C, TRAY)
display(HTML(A2.to_jshtml()))
print("Guardado:", guardar_animacion(A2, "A2_vorticidad_particulas"), f"({time.time() - t0:.0f} s)")

### A3 — Líneas de corriente (se recalculan en cada marco)

In [ ]:
def anim_corrientes(C, fps=8, paso=4):
    fig, a = plt.subplots(figsize=(9, 4.2))
    def frame(k):
        a.clear()
        mapa(a, C, C.rapidez(k), k, "rapidez", vlim=(0, vmax_sp), barra=False)
        corrientes(a, C, k, densidad=1.0, color="white")
        preparar_eje(a, C, f"Líneas de corriente — t = {C.t[k]:.1f} s")
        return []
    an = animation.FuncAnimation(fig, frame, frames=range(0, len(C), paso), interval=1000 / fps)
    plt.close(fig)
    return an

t0 = time.time()
A3 = anim_corrientes(C)
display(HTML(A3.to_jshtml()))
print("Guardado:", guardar_animacion(A3, "A3_corrientes", fps=8), f"({time.time() - t0:.0f} s)")

## 5. Resumen del proyecto en una figura: el registro de validación

In [ ]:
if os.path.exists("registro_validacion.csv"):
    REG = pd.read_csv("registro_validacion.csv")
    print("Usando registro_validacion.csv")
else:
    REG = pd.DataFrame([  # valores de las ejecuciones de las fases 1b y 4
        ("1a · tampón irreversible", "consenso", "retención 1 h [%]", 32.08, 6.28, 49.22, 37, 90),
        ("1a · tampón irreversible", "consenso", "retención 2 h [%]", 1.90, 0.47, 18.78, 30, 60),
        ("1a · tampón irreversible", "consenso", "retención 4 h [%]", 0.28, 0.04, 1.06, 0, 10),
        ("1a · tampón irreversible", "1000 kcal", "pH máximo", 5.06, 4.56, 5.32, 5.8, 6.7),
        ("1b · tampón en equilibrio", "consenso", "retención 1 h [%]", 32.08, 6.28, 49.22, 37, 90),
        ("1b · tampón en equilibrio", "consenso", "retención 2 h [%]", 1.90, 0.47, 18.78, 30, 60),
        ("1b · tampón en equilibrio", "consenso", "retención 4 h [%]", 0.28, 0.04, 1.06, 0, 10),
        ("1b · tampón en equilibrio", "1000 kcal", "pH máximo", 5.93, 5.49, 6.35, 5.8, 6.7),
        ("4 · partículas + criba", "consenso", "retención 1 h [%]", 72.89, 38.86, 91.93, 37, 90),
        ("4 · partículas + criba", "consenso", "retención 2 h [%]", 24.20, 10.0, 70.59, 30, 60),
        ("4 · partículas + criba", "consenso", "retención 4 h [%]", 7.68, 3.73, 30.99, 0, 10),
    ], columns=["versión", "comida", "métrica", "nominal", "p5", "p95", "objetivo_min", "objetivo_max"])
    print("No hay registro_validacion.csv: se usan los valores de las ejecuciones de las fases 1b y 4")

versiones = list(dict.fromkeys(REG["versión"]))
col_v = dict(zip(versiones, [C5, C3, C1, C2, C4]))
metricas = list(dict.fromkeys(REG["métrica"]))
fig, axs = plt.subplots(1, len(metricas), figsize=(3.2 * len(metricas), 4.0))
for a, met in zip(np.atleast_1d(axs), metricas):
    sub = REG[REG["métrica"] == met]
    lo, hi = sub["objetivo_min"].iloc[0], sub["objetivo_max"].iloc[0]
    a.axhspan(lo, hi, color=GRIS_MEDIO, alpha=0.25, lw=0)
    for j, v in enumerate(versiones):
        f = sub[sub["versión"] == v]
        if f.empty: continue
        f = f.iloc[0]
        a.errorbar(j, f["nominal"], yerr=[[f["nominal"] - f["p5"]], [f["p95"] - f["nominal"]]], fmt="o", ms=7, capsize=4, color=col_v[v])
    a.set_xticks(range(len(versiones))); a.set_xticklabels([v.split(" · ")[0] for v in versiones])
    a.set_xlim(-0.6, len(versiones) - 0.4); a.set_title(met, fontsize=10); a.grid(axis="x", visible=False)
fig.legend(handles=[plt.Line2D([], [], marker="o", ls="none", color=col_v[v], label=v) for v in versiones] +
           [plt.Rectangle((0, 0), 1, 1, color=GRIS_MEDIO, alpha=0.35, label="rango medido")],
           loc="upper center", ncol=len(versiones) + 1, bbox_to_anchor=(0.5, 1.05), fontsize=9)
fig.suptitle("Registro de validación: cada versión del modelo frente a los datos", y=1.12, fontweight="bold")
fig.tight_layout(); guardar_figura(fig, "05_registro_validacion"); plt.show()

## 6. Plotly: figuras interactivas (opcional)

Plotly viene instalado en Colab. Las figuras se pueden ampliar, permiten consultar valores pasando el ratón y se exportan a HTML autónomo para compartir. Si Plotly no está disponible, la sección se salta sin error.

- **P1**: mapa de rapidez con botón de reproducción y deslizador temporal.
- **P2**: trayectorias de partículas, con su tamaño y tiempo al pasar el ratón.
- **P3**: el registro de validación con barras de incertidumbre.

In [ ]:
try:
    import plotly.graph_objects as go
    HAY_PLOTLY = True
except ImportError:
    HAY_PLOTLY = False
    print("Plotly no está instalado: pip install plotly  (en Colab ya viene)")

def plotly_mapa_animado(C, sub=2, cada_marco=2):
    s = (slice(None, None, sub), slice(None, None, sub))
    ks = list(range(0, len(C), cada_marco))
    z = lambda k: np.round(C.rapidez(k)[s], 4)
    fig = go.Figure(
        data=[go.Heatmap(z=z(ks[0]), x=C.x[::sub], y=C.y[::sub], colorscale="Magma", zmin=0, zmax=vmax_sp,
                         colorbar=dict(title="cm/s"), hovertemplate="x=%{x:.2f} cm<br>y=%{y:.2f} cm<br>|u|=%{z:.3f} cm/s<extra></extra>")],
        frames=[go.Frame(data=[go.Heatmap(z=z(k))], name=f"{C.t[k]:.1f}") for k in ks])
    pasos = [dict(method="animate", label=f"{C.t[k]:.0f}", args=[[f"{C.t[k]:.1f}"], dict(mode="immediate", frame=dict(duration=0, redraw=True))]) for k in ks]
    fig.update_layout(
        title="Rapidez del flujo en el antro (P1)", width=900, height=480,
        xaxis=dict(title="x [cm]", scaleanchor="y", constrain="domain"), yaxis=dict(title="y [cm]"),
        updatemenus=[dict(type="buttons", x=0.0, y=-0.12, xanchor="left", buttons=[
            dict(label="▶ reproducir", method="animate", args=[None, dict(frame=dict(duration=120, redraw=True), fromcurrent=True)]),
            dict(label="⏸ pausa", method="animate", args=[[None], dict(mode="immediate", frame=dict(duration=0, redraw=False))])])],
        sliders=[dict(active=0, y=-0.05, currentvalue=dict(prefix="t = ", suffix=" s"), steps=pasos)])
    return fig

def plotly_trayectorias(TRAY, n_por_tamano=15):
    fig = go.Figure()
    for dv in COL_D:
        for i, q in enumerate(np.where(TRAY["d_mm"] == dv)[0][:n_por_tamano]):
            fig.add_trace(go.Scatter(x=TRAY["x"][:, q], y=TRAY["y"][:, q], mode="lines", line=dict(color=COL_D[dv], width=1.2),
                                     name=f"{dv} mm", legendgroup=str(dv), showlegend=(i == 0), customdata=TRAY["t"],
                                     hovertemplate=f"d = {dv} mm<br>t = %{{customdata:.1f}} s<br>x = %{{x:.2f}}, y = %{{y:.2f}} cm<extra></extra>"))
    fig.update_layout(title="Trayectorias de partículas (P2)", width=900, height=450, template="plotly_white",
                      xaxis=dict(title="x [cm]", scaleanchor="y"), yaxis=dict(title="y [cm]"))
    return fig

def plotly_registro(REG):
    fig = go.Figure()
    for v in versiones:
        f = REG[REG["versión"] == v]
        fig.add_trace(go.Scatter(x=f["métrica"], y=f["nominal"], mode="markers", name=v, marker=dict(size=10, color=col_v[v]),
                                 error_y=dict(type="data", symmetric=False, array=f["p95"] - f["nominal"], arrayminus=f["nominal"] - f["p5"]),
                                 hovertemplate="%{x}<br>nominal %{y:.2f}<extra>" + v + "</extra>"))
    for _, r in REG.drop_duplicates("métrica").iterrows():
        fig.add_shape(type="rect", xref="x", yref="y", x0=list(metricas).index(r["métrica"]) - 0.4, x1=list(metricas).index(r["métrica"]) + 0.4,
                      y0=r["objetivo_min"], y1=r["objetivo_max"], fillcolor="lightgray", opacity=0.4, line_width=0, layer="below")
    fig.update_layout(title="Registro de validación (P3) — rectángulo gris: rango medido", width=900, height=450,
                      template="plotly_white", yaxis=dict(title="valor de la métrica"))
    return fig

if HAY_PLOTLY:
    os.makedirs("figuras", exist_ok=True)
    for nombre, fabrica in (("P1_mapa_animado", lambda: plotly_mapa_animado(C)), ("P2_trayectorias", lambda: plotly_trayectorias(TRAY)),
                            ("P3_registro", lambda: plotly_registro(REG))):
        try:
            fig = fabrica(); fig.show(); fig.write_html(f"figuras/{nombre}.html", include_plotlyjs="cdn")
            print(f"figuras/{nombre}.html")
        except Exception as e:
            print(f"{nombre}: no se pudo generar ({type(e).__name__}: {e})")

## 7. Exportación y comprobaciones finales

In [ ]:
archivos = sorted(glob.glob("figuras/*"))
tabla = pd.DataFrame([{"archivo": f, "tamaño [MB]": os.path.getsize(f) / 1e6} for f in archivos])
display(tabla.style.format({"tamaño [MB]": "{:.2f}"}).hide(axis="index"))
for ext in ("png", "svg"):
    n = len([f for f in archivos if f.endswith(ext)])
    VERIFICACION.append({"test": f"V2 figuras .{ext} exportadas", "valor": n, "criterio": "≥ 5", "pasa": n >= 5})
anims = [f for f in archivos if f.endswith((".mp4", ".gif"))]
VERIFICACION.append({"test": "V3 animaciones guardadas (mp4/gif)", "valor": len(anims), "criterio": "≥ 3", "pasa": len(anims) >= 3})

import shutil
shutil.make_archive("figuras_estomago", "zip", "figuras")
print(f"figuras_estomago.zip: {os.path.getsize('figuras_estomago.zip') / 1e6:.1f} MB")
try:
    from google.colab import files     # en Colab, descarga el zip
    files.download("figuras_estomago.zip")
except ImportError:
    pass

ver = pd.DataFrame(VERIFICACION)
ver["resultado"] = np.where(ver["pasa"], "✅ pasa", "❌ FALLA")
display(ver.drop(columns="pasa").style.format({"valor": "{:.3g}"}).hide(axis="index"))
assert ver["pasa"].all(), "Hay comprobaciones que no pasan"

## 8. Cierre del proyecto

| Fase | Qué resuelve | Herramienta clave | Validación / verificación |
|---|---|---|---|
| 1a | Química 0D (ácido, pepsina, proteína) | Integradores propios, rigidez | Órdenes de convergencia, conservación |
| 1b | Química con vaciado, almidón, tampón en equilibrio | Radau + Monte Carlo paramétrico | Retención gástrica y pH frente a datos clínicos |
| 2 | Transporte 1D/2D | Volúmenes finitos MUSCL | Solución manufacturada, regresión al 0D |
| 3 | Navier–Stokes con paredes móviles | Lattice Boltzmann (NumPy/CuPy/JAX) | Poiseuille, Couette, Taylor–Green, Shapiro |
| 4 | Partículas: arrastre, rotura, erosión, criba | Balance de poblaciones + Monte Carlo | Soluciones exactas; el error de modelo baja de 0.34 a 0.06 |
| 5 | Visualización | Matplotlib + Plotly | Campos derivados verificados, exportación |

**Siguiente paso natural.** El modelo integrado en un único bucle de tiempo: flujo LBM, transporte químico y partículas lagrangianas, con el píloro abierto. Ahora sí, con todas las piezas verificadas por separado y un registro de validación para medir si la integración mejora el acuerdo con los datos.